ENVIRONMENT SETUP & DATASET LOADING

In [1]:
from pathlib import Path
import random
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

# 1. Reproducibility
SEED = 7

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate project root
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (
            p / "experiments" / "05_ablation"
            / "data" / "ablation_manifest.csv"
        ).is_file()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate ablation_manifest.csv. "
        "Check Cell 10 of 01_ablation_data.ipynb."
    )

EXPERIMENT_DIR = (
    PROJECT_ROOT / "experiments" / "05_ablation"
)

DATA_DIR = EXPERIMENT_DIR / "data"

# 3. Dataset files
DATASET_FILES = {
    "Electricity": "electricity_ablation.pt",
    "Financial": "financial_ablation.pt",
    "CMIP6 Wind": "cmip6_wind_ablation.pt",
    "CMIP6 Pressure": "cmip6_pressure_candidate_2500.pt",
}

# 4. Load validated datasets
ABLATION_DATASETS = {}

for name, filename in DATASET_FILES.items():

    path = DATA_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    bundle = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    train = bundle["train"]
    test = bundle["test"]

    assert train.ndim == 2
    assert test.ndim == 2
    assert train.shape[1] == test.shape[1]
    assert train.shape[1] == bundle["input_dim"]

    assert torch.isfinite(train).all()
    assert torch.isfinite(test).all()

    ABLATION_DATASETS[name] = bundle

# 5. Ablation grid
PATCH_LENGTHS = [70, 80, 90, 100, 110, 120, 130]
HORIZONS = [2, 4, 6, 8, 10, 12, 14, 16]

# 6. Model hyperparameters
D_MODEL = 96
D_LATENT = 96
NUM_ENCODER_LAYERS = 3
NUM_ATTENTION_HEADS = 4
DIM_FEEDFORWARD = 96
LSTM_HIDDEN_DIM = 96
LSTM_NUM_LAYERS = 2

RHO_MAX = 0.99
LYAP_WEIGHT = 0.1

EPOCHS = 4000
LEARNING_RATE = 3e-4

# 7. Six ablation variants
MODEL_VARIANTS = [
    "DLinear",
    "LSTM",
    "PatchTST",
    "PatchTST+LinearLatent",
    "KoopPatchTST_NoLyap",
    "KoopPatchTST_Full",
]

# 8. Dataset summary
summary = []

for name, bundle in ABLATION_DATASETS.items():

    summary.append({
        "Dataset": name,
        "Channels": bundle["input_dim"],
        "Train": tuple(bundle["train"].shape),
        "Test": tuple(bundle["test"].shape),
        "Status": bundle["status"]
    })

summary_df = pd.DataFrame(summary)

# 9. Final output
print("=== ABLATION MODEL ENVIRONMENT ===")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Seed:", SEED)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n=== DATASETS ===")
display(summary_df)

print("\n=== MODEL CONFIGURATION ===")
print("Models:", len(MODEL_VARIANTS))
print("Model names:", MODEL_VARIANTS)

print("d_model:", D_MODEL)
print("d_latent:", D_LATENT)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Attention heads:", NUM_ATTENTION_HEADS)
print("Feedforward dim:", DIM_FEEDFORWARD)

print("LSTM hidden dim:", LSTM_HIDDEN_DIM)
print("LSTM layers:", LSTM_NUM_LAYERS)

print("rho_max:", RHO_MAX)
print("Lyapunov weight:", LYAP_WEIGHT)

print("\n=== TRAINING CONFIGURATION ===")
print("Epochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)

print("Patch lengths:", PATCH_LENGTHS)
print("Horizons:", HORIZONS)

print("\nTotal dataset-grid combinations:",
      len(ABLATION_DATASETS)
      * len(PATCH_LENGTHS)
      * len(HORIZONS))

print("Total model configurations:",
      len(ABLATION_DATASETS)
      * len(PATCH_LENGTHS)
      * len(HORIZONS)
      * len(MODEL_VARIANTS))

print("\nEnvironment setup completed!")


=== ABLATION MODEL ENVIRONMENT ===
Project root: c:\Users\Win 11 New\Desktop\pa gk
Device: cuda
PyTorch: 2.0.1+cu118
Seed: 7
GPU: Quadro T1000

=== DATASETS ===


c:\Users\Win 11 New\Desktop\pa gk\.venv\lib\site-packages\torch\_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


,Dataset,Channels,Train,Test,Status
0,Electricity,6,"(2800, 6)","(700, 6)",verified_local_pipeline
1,Financial,2,"(2320, 2)","(580, 2)",verified_local_pipeline
2,CMIP6 Wind,5,"(2336, 5)","(584, 5)",verified_local_extraction
3,CMIP6 Pressure,5,"(2000, 5)","(500, 5)",candidate_2500



=== MODEL CONFIGURATION ===
Models: 6
Model names: ['DLinear', 'LSTM', 'PatchTST', 'PatchTST+LinearLatent', 'KoopPatchTST_NoLyap', 'KoopPatchTST_Full']
d_model: 96
d_latent: 96
Encoder layers: 3
Attention heads: 4
Feedforward dim: 96
LSTM hidden dim: 96
LSTM layers: 2
rho_max: 0.99
Lyapunov weight: 0.1

=== TRAINING CONFIGURATION ===
Epochs: 4000
Learning rate: 0.0003
Patch lengths: [70, 80, 90, 100, 110, 120, 130]
Horizons: [2, 4, 6, 8, 10, 12, 14, 16]

Total dataset-grid combinations: 224
Total model configurations: 1344

Environment setup completed!


KOOPMAN & LINEAR LATENT OPERATORS

In [2]:
# 1. Orthogonalization
def _orth(w: torch.Tensor) -> torch.Tensor:
    return torch.linalg.qr(w)[0]


# 2. Strictly Stable Koopman Operator
class StrictStableKoopmanOperator(nn.Module):
    """
    Orthogonal-Diagonal-Orthogonal (ODO)
    parameterization with singular values < rho_max.
    """

    def __init__(
        self,
        latent_dim: int,
        ρ_max: float = 0.99
    ):
        super().__init__()

        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_raw = nn.Parameter(
            torch.randn(latent_dim)
        )

        self.ρ_max = ρ_max

    def forward(self, z: torch.Tensor):

        U = _orth(self.U_raw)
        V = _orth(self.V_raw)

        Σ = torch.sigmoid(self.S_raw) * self.ρ_max

        K = U @ torch.diag(Σ) @ V.T

        z_next = z @ K.T

        return z_next, K, Σ


# 3. Unconstrained Linear Latent Propagator
class LinearLatentPropagator(nn.Module):
    """
    Unconstrained latent transformation:
        z_next = W(z)

    Used by PatchTST+LinearLatent ablation.
    """

    def __init__(self, latent_dim: int):
        super().__init__()

        self.W = nn.Linear(
            latent_dim,
            latent_dim,
            bias=False
        )

    def forward(self, z: torch.Tensor):

        K = self.W.weight.T
        Σ = None

        return self.W(z), K, Σ


# 4. Initialize operators for validation
latent_dim = D_LATENT

koopman = StrictStableKoopmanOperator(
    latent_dim=latent_dim,
    ρ_max=RHO_MAX
)

linear_latent = LinearLatentPropagator(
    latent_dim=latent_dim
)

# 5. Create sample latent representations
z = torch.randn(4, latent_dim)

# 6. Forward pass
with torch.no_grad():

    z_koop, K_koop, sigma = koopman(z)

    z_linear, K_linear, sigma_linear = (
        linear_latent(z)
    )

# 7. Shape validation
assert z_koop.shape == z.shape
assert z_linear.shape == z.shape

assert K_koop.shape == (latent_dim, latent_dim)
assert K_linear.shape == (latent_dim, latent_dim)

assert sigma.shape == (latent_dim,)
assert sigma_linear is None

# 8. Validate Koopman operator
singular_values = torch.linalg.svdvals(K_koop)

assert torch.max(singular_values) < RHO_MAX + 1e-5

assert torch.allclose(
    z_koop,
    z @ K_koop.T,
    atol=1e-5
)

# 9. Validate Linear Latent operator
assert torch.allclose(
    z_linear,
    linear_latent.W(z),
    atol=1e-6
)

# 10. Parameter counts
koop_params = sum(
    p.numel() for p in koopman.parameters()
)

linear_params = sum(
    p.numel() for p in linear_latent.parameters()
)

# 11. Summary
print("=== LATENT OPERATOR VALIDATION ===")

print("\nInput latent shape:", z.shape)

print("\n--- Strict Stable Koopman ---")
print("Output shape:", z_koop.shape)
print("Operator K:", K_koop.shape)
print("Sigma shape:", sigma.shape)

print("Max singular value:",
      singular_values.max().item())

print("Spectral radius upper bound:", RHO_MAX)
print("Parameters:", koop_params)

print("\n--- Linear Latent Propagator ---")
print("Output shape:", z_linear.shape)
print("Operator shape:", K_linear.shape)
print("Parameters:", linear_params)

print("\nBoth operator forward passes validated!")


=== LATENT OPERATOR VALIDATION ===

Input latent shape: torch.Size([4, 96])

--- Strict Stable Koopman ---
Output shape: torch.Size([4, 96])
Operator K: torch.Size([96, 96])
Sigma shape: torch.Size([96])
Max singular value: 0.9516687393188477
Spectral radius upper bound: 0.99
Parameters: 18528

--- Linear Latent Propagator ---
Output shape: torch.Size([4, 96])
Operator shape: torch.Size([96, 96])
Parameters: 9216

Both operator forward passes validated!


POSITIONAL ENCODING & PATCH EMBEDDING

In [3]:
# 1. Sinusoidal Positional Encoding
class SinCosPosEnc(nn.Module):

    def __init__(
        self,
        d_model: int,
        max_len: int = 10_000
    ):
        super().__init__()

        pos = torch.arange(
            max_len
        ).float().unsqueeze(1)

        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10_000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]


# 2. One-dimensional Patch Embedding
class PatchEmbed1D(nn.Module):

    def __init__(
        self,
        in_ch: int,
        d_model: int,
        patch_len: int,
        stride: int
    ):
        super().__init__()

        self.conv = nn.Conv1d(
            in_ch,
            d_model,
            kernel_size=patch_len,
            stride=stride
        )

    def forward(self, x):

        # (B, T, C) -> (B, C, T)
        x = x.permute(0, 2, 1)

        # (B, C, T) -> (B, d_model, P)
        x = self.conv(x)

        # (B, d_model, P) -> (B, P, d_model)
        return x.permute(0, 2, 1)


# 3. Test with an ablation configuration
BATCH_SIZE = 4
SEQ_LEN = 120
INPUT_DIM = 5
PATCH_LEN = 120
STRIDE = PATCH_LEN

x = torch.randn(
    BATCH_SIZE,
    SEQ_LEN,
    INPUT_DIM
)

# 4. Patch Embedding
patch_embedding = PatchEmbed1D(
    in_ch=INPUT_DIM,
    d_model=D_MODEL,
    patch_len=PATCH_LEN,
    stride=STRIDE
)

patches = patch_embedding(x)

expected_patches = (
    (SEQ_LEN - PATCH_LEN) // STRIDE + 1
)

assert patches.shape == (
    BATCH_SIZE,
    expected_patches,
    D_MODEL
)

# 5. Positional Encoding
pos_encoding = SinCosPosEnc(
    d_model=D_MODEL,
    max_len=expected_patches + 1
)

encoded_patches = pos_encoding(patches)

assert encoded_patches.shape == patches.shape

# 6. Validate positional encoding
assert torch.allclose(
    encoded_patches,
    patches + pos_encoding.pe[:expected_patches],
    atol=1e-6
)

# 7. Additional multi-patch test
multi_patch_embedding = PatchEmbed1D(
    in_ch=INPUT_DIM,
    d_model=D_MODEL,
    patch_len=30,
    stride=30
)

multi_patches = multi_patch_embedding(x)

assert multi_patches.shape == (
    BATCH_SIZE, 4, D_MODEL
)

# 8. Parameter validation
patch_params = sum(
    p.numel()
    for p in patch_embedding.parameters()
)

pos_params = sum(
    p.numel()
    for p in pos_encoding.parameters()
)

# 9. Summary
print("=== PATCH EMBEDDING VALIDATION ===")
print("Input shape:", x.shape)
print("Patch length:", PATCH_LEN)
print("Stride:", STRIDE)

print("\nNumber of patches:", expected_patches)
print("Patch embedding output:", patches.shape)
print("Embedding parameters:", patch_params)

print("\n=== POSITIONAL ENCODING VALIDATION ===")
print("Encoded output:", encoded_patches.shape)
print("Trainable parameters:", pos_params)

print("\n=== MULTI-PATCH TEST ===")
print("Patch length: 30")
print("Output:", multi_patches.shape)

print("\nPatch Embedding validated.")
print("Positional Encoding validated.")
print("Cell 3 completed successfully!")


=== PATCH EMBEDDING VALIDATION ===
Input shape: torch.Size([4, 120, 5])
Patch length: 120
Stride: 120

Number of patches: 1
Patch embedding output: torch.Size([4, 1, 96])
Embedding parameters: 57696

=== POSITIONAL ENCODING VALIDATION ===
Encoded output: torch.Size([4, 1, 96])
Trainable parameters: 0

=== MULTI-PATCH TEST ===
Patch length: 30
Output: torch.Size([4, 4, 96])

Patch Embedding validated.
Positional Encoding validated.
Cell 3 completed successfully!


PATCHTST BACKBONE

In [4]:
# 1. PatchTST Backbone
class PatchTST_Backbone(nn.Module):

    def __init__(
        self,
        input_dim: int,
        seq_len: int,
        patch_len: int,
        d_model: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dim_ff: int = 96
    ):
        super().__init__()

        # Patch Embedding
        self.patch = PatchEmbed1D(
            input_dim,
            d_model,
            patch_len,
            patch_len
        )

        # Positional Encoding
        n_patches = int(
            np.ceil(seq_len / patch_len)
        )

        self.pos = SinCosPosEnc(
            d_model,
            max_len=n_patches + 1
        )

        # Transformer Encoder
        enc = nn.TransformerEncoderLayer(
            d_model,
            num_heads,
            dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            enc,
            num_layers
        )

        # Learnable CLS token
        self.cls = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

    def forward(self, x):

        # (B, T, C) -> (B, P, d_model)
        x = self.patch(x)

        # Expand CLS token for the batch
        cls = self.cls.expand(
            x.size(0), -1, -1
        )

        # (B, P, d_model) -> (B, 1+P, d_model)
        x = torch.cat(
            [cls, x],
            dim=1
        )

        # Transformer Encoder
        x = self.encoder(
            self.pos(x)
        )

        # Return CLS representation
        return x[:, 0]


# 2. Test configuration
TEST_BATCH_SIZE = 4
TEST_INPUT_DIM = 5
TEST_PATCH_LEN = 120

backbone = PatchTST_Backbone(
    input_dim=TEST_INPUT_DIM,
    seq_len=TEST_PATCH_LEN,
    patch_len=TEST_PATCH_LEN,
    d_model=D_MODEL,
    num_layers=NUM_ENCODER_LAYERS,
    num_heads=NUM_ATTENTION_HEADS,
    dim_ff=DIM_FEEDFORWARD
)

# 3. Sample input
x_test = torch.randn(
    TEST_BATCH_SIZE,
    TEST_PATCH_LEN,
    TEST_INPUT_DIM
)

# 4. Forward pass
backbone.eval()

with torch.no_grad():
    latent = backbone(x_test)

# 5. Validate output
assert latent.shape == (
    TEST_BATCH_SIZE,
    D_MODEL
)

assert torch.isfinite(latent).all()

# 6. Validate backbone components
assert len(backbone.encoder.layers) == 3

assert backbone.cls.shape == (
    1, 1, D_MODEL
)

assert backbone.patch.conv.kernel_size == (
    TEST_PATCH_LEN,
)

assert backbone.patch.conv.stride == (
    TEST_PATCH_LEN,
)

# 7. Count trainable parameters
backbone_params = sum(
    p.numel()
    for p in backbone.parameters()
    if p.requires_grad
)

# 8. Check other ablation datasets
test_configurations = {
    "Electricity": (6, 70),
    "Financial": (2, 80),
    "CMIP6 Wind": (5, 120),
    "CMIP6 Pressure": (5, 130)
}

validation_results = []

for name, (input_dim, patch_len) in (
    test_configurations.items()
):

    model = PatchTST_Backbone(
        input_dim=input_dim,
        seq_len=patch_len,
        patch_len=patch_len,
        d_model=D_MODEL,
        num_layers=NUM_ENCODER_LAYERS,
        num_heads=NUM_ATTENTION_HEADS,
        dim_ff=DIM_FEEDFORWARD
    )

    model.eval()

    dummy_input = torch.randn(
        TEST_BATCH_SIZE,
        patch_len,
        input_dim
    )

    with torch.no_grad():
        output = model(dummy_input)

    assert output.shape == (
        TEST_BATCH_SIZE,
        D_MODEL
    )

    assert torch.isfinite(output).all()

    validation_results.append({
        "Dataset": name,
        "Input": tuple(dummy_input.shape),
        "Output": tuple(output.shape),
        "Status": "PASSED"
    })

# 9. Final summary
print("=== PATCHTST BACKBONE VALIDATION ===")

print("Input shape:", x_test.shape)
print("Latent output shape:", latent.shape)

print("\n=== ARCHITECTURE ===")
print("d_model:", D_MODEL)
print("Encoder layers:", len(backbone.encoder.layers))
print("Attention heads:", NUM_ATTENTION_HEADS)
print("Feedforward dimension:", DIM_FEEDFORWARD)

print("CLS token shape:", backbone.cls.shape)
print("Trainable parameters:", backbone_params)

print("\n=== MULTI-DATASET VALIDATION ===")
display(pd.DataFrame(validation_results))

print("\nAll backbone forward passes validated!")
print("Cell 4 completed successfully!")


=== PATCHTST BACKBONE VALIDATION ===
Input shape: torch.Size([4, 120, 5])
Latent output shape: torch.Size([4, 96])

=== ARCHITECTURE ===
d_model: 96
Encoder layers: 3
Attention heads: 4
Feedforward dimension: 96
CLS token shape: torch.Size([1, 1, 96])
Trainable parameters: 226560

=== MULTI-DATASET VALIDATION ===


,Dataset,Input,Output,Status
0,Electricity,"(4, 70, 6)","(4, 96)",PASSED
1,Financial,"(4, 80, 2)","(4, 96)",PASSED
2,CMIP6 Wind,"(4, 120, 5)","(4, 96)",PASSED
3,CMIP6 Pressure,"(4, 130, 5)","(4, 96)",PASSED



All backbone forward passes validated!
Cell 4 completed successfully!


PATCHTST ABLATION VARIANTS

In [5]:
# 1. Standard PatchTST Baseline
class PatchTSTBaseline(nn.Module):
    """
    PatchTST backbone with direct forecasting head.
    No latent propagation operator.
    """

    def __init__(
        self, input_dim, seq_len, horizon,
        patch_len, d_model=64
    ):
        super().__init__()

        self.backbone = PatchTST_Backbone(
            input_dim,
            seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)
        pred = self.fc(z)

        if return_latents:
            return pred, z, z

        return pred


# 2. PatchTST + Unconstrained Linear Latent
class PatchTST_UnconstrainedLatent(nn.Module):
    """
    PatchTST with an unconstrained linear latent
    transformation and no Lyapunov regularization.
    """

    def __init__(
        self, input_dim, seq_len, horizon,
        patch_len, d_model=64
    ):
        super().__init__()

        self.backbone = PatchTST_Backbone(
            input_dim,
            seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        self.koop = LinearLatentPropagator(d_model)

        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)

        z_next, _, _ = self.koop(z)

        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


# 3. PatchTST + Strict Stable Koopman
class Koopformer_PatchTST(nn.Module):
    """
    PatchTST with strictly stable Koopman operator.

    Used for:
        KoopPatchTST_NoLyap (lambda = 0.0)
        KoopPatchTST_Full   (lambda = 0.1)
    """

    def __init__(
        self, input_dim, seq_len, horizon,
        patch_len, d_model=64
    ):
        super().__init__()

        self.backbone = PatchTST_Backbone(
            input_dim,
            seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        self.koop = StrictStableKoopmanOperator(
            d_model
        )

        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        z = self.backbone(x)

        z_next, _, _ = self.koop(z)

        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


# 4. Test configuration
TEST_BATCH = 4
TEST_INPUT_DIM = 5
TEST_PATCH_LEN = 120
TEST_HORIZON = 8

dummy_x = torch.randn(
    TEST_BATCH,
    TEST_PATCH_LEN,
    TEST_INPUT_DIM
)

# 5. Initialize all PatchTST variants
patchtst_variants = {
    "PatchTST": PatchTSTBaseline(
        TEST_INPUT_DIM,
        TEST_PATCH_LEN,
        TEST_HORIZON,
        patch_len=TEST_PATCH_LEN,
        d_model=D_MODEL
    ),

    "PatchTST+LinearLatent": PatchTST_UnconstrainedLatent(
        TEST_INPUT_DIM,
        TEST_PATCH_LEN,
        TEST_HORIZON,
        patch_len=TEST_PATCH_LEN,
        d_model=D_MODEL
    ),

    "KoopPatchTST_NoLyap": Koopformer_PatchTST(
        TEST_INPUT_DIM,
        TEST_PATCH_LEN,
        TEST_HORIZON,
        patch_len=TEST_PATCH_LEN,
        d_model=D_MODEL
    ),

    "KoopPatchTST_Full": Koopformer_PatchTST(
        TEST_INPUT_DIM,
        TEST_PATCH_LEN,
        TEST_HORIZON,
        patch_len=TEST_PATCH_LEN,
        d_model=D_MODEL
    )
}

# 6. Validate forward passes
variant_results = []

for name, model in patchtst_variants.items():

    model.eval()

    with torch.no_grad():
        pred, z, z_next = model(
            dummy_x,
            return_latents=True
        )

        direct_pred = model(dummy_x)

    # Forecast output
    assert pred.shape == (
        TEST_BATCH,
        TEST_INPUT_DIM * TEST_HORIZON
    )

    # Latent representations
    assert z.shape == (TEST_BATCH, D_MODEL)
    assert z_next.shape == (TEST_BATCH, D_MODEL)

    # Check both forward interfaces agree
    torch.testing.assert_close(
        pred, direct_pred
    )

    assert torch.isfinite(pred).all()
    assert torch.isfinite(z).all()
    assert torch.isfinite(z_next).all()

    # Validate operator type
    if name == "PatchTST":
        assert not hasattr(model, "koop")
        torch.testing.assert_close(z, z_next)

    elif name == "PatchTST+LinearLatent":
        assert isinstance(
            model.koop,
            LinearLatentPropagator
        )

        with torch.no_grad():
            expected_z_next = model.koop.W(z)

        torch.testing.assert_close(
            z_next, expected_z_next
        )

    else:
        assert isinstance(
            model.koop,
            StrictStableKoopmanOperator
        )

        with torch.no_grad():
            _, K, sigma = model.koop(z)

        assert sigma.max().item() < RHO_MAX
        assert torch.linalg.svdvals(K).max().item() < (
            RHO_MAX + 1e-5
        )

    n_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    variant_results.append({
        "Model": name,
        "Prediction": tuple(pred.shape),
        "Latent": tuple(z.shape),
        "Parameters": n_params,
        "Status": "PASSED"
    })


# 7. Validate parameter count differences
parameter_counts = {
    row["Model"]: row["Parameters"]
    for row in variant_results
}

assert (
    parameter_counts["PatchTST+LinearLatent"]
    - parameter_counts["PatchTST"]
) == D_MODEL ** 2

assert (
    parameter_counts["KoopPatchTST_NoLyap"]
    - parameter_counts["PatchTST"]
) == 2 * D_MODEL ** 2 + D_MODEL

assert (
    parameter_counts["KoopPatchTST_NoLyap"]
    == parameter_counts["KoopPatchTST_Full"]
)

# 8. Final summary
print("=== PATCHTST ABLATION VALIDATION ===")

print("Input:", tuple(dummy_x.shape))
print("Forecast horizon:", TEST_HORIZON)
print("Number of channels:", TEST_INPUT_DIM)
print("Latent dimension:", D_MODEL)

print("\n=== MODEL VARIANTS ===")
display(pd.DataFrame(variant_results))

print("\n=== LYAPUNOV CONFIGURATION ===")
print("KoopPatchTST_NoLyap: lambda = 0.0")
print("KoopPatchTST_Full:   lambda = 0.1")

print("\nAll PatchTST variants validated!")
print("Cell 5 completed successfully!")


=== PATCHTST ABLATION VALIDATION ===
Input: (4, 120, 5)
Forecast horizon: 8
Number of channels: 5
Latent dimension: 96

=== MODEL VARIANTS ===


,Model,Prediction,Latent,Parameters,Status
0,PatchTST,"(4, 40)","(4, 96)",230440,PASSED
1,PatchTST+LinearLatent,"(4, 40)","(4, 96)",239656,PASSED
2,KoopPatchTST_NoLyap,"(4, 40)","(4, 96)",248968,PASSED
3,KoopPatchTST_Full,"(4, 40)","(4, 96)",248968,PASSED



=== LYAPUNOV CONFIGURATION ===
KoopPatchTST_NoLyap: lambda = 0.0
KoopPatchTST_Full:   lambda = 0.1

All PatchTST variants validated!
Cell 5 completed successfully!


LSTM & DLINEAR BASELINES

In [6]:
# 1. LSTM Baseline
class SimpleLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        horizon
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        _, (h_n, _) = self.lstm(x)

        z = h_n[-1]

        out = self.fc(z)

        if return_latents:
            return out, z, z

        return out


# 2. DLinear Baseline — Author's implementation
class DLinearForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon
    ):
        super().__init__()

        self.linears = nn.ModuleList([
            nn.Conv1d(
                1, 1,
                kernel_size=seq_len,
                bias=True
            )
            for _ in range(input_dim)
        ])

        self.horizon = horizon
        self.seq_len = seq_len
        self.input_dim = input_dim

    def forward(self, x, return_latents=False):

        outs = []

        for i, linear in enumerate(self.linears):

            xi = x[:, :, i].unsqueeze(1)

            if xi.shape[-1] < self.seq_len:
                pad = self.seq_len - xi.shape[-1]
                xi = torch.nn.functional.pad(
                    xi, (pad, 0)
                )

            yi = linear(xi)

            if yi.shape[-1] == 1 and self.horizon > 1:
                yi = yi.expand(
                    -1, -1, self.horizon
                )

            outs.append(yi)

        out = torch.cat(outs, dim=1)

        out = out.permute(
            0, 2, 1
        ).reshape(x.size(0), -1)

        if return_latents:
            return out, out, out

        return out


# 3. Test configuration
BASELINE_BATCH = 4
BASELINE_INPUT_DIM = 5
BASELINE_PATCH_LEN = 120
BASELINE_HORIZON = 8

baseline_input = torch.randn(
    BASELINE_BATCH,
    BASELINE_PATCH_LEN,
    BASELINE_INPUT_DIM
)

# 4. Initialize baselines
baseline_models = {
    "DLinear": DLinearForecaster(
        input_dim=BASELINE_INPUT_DIM,
        seq_len=BASELINE_PATCH_LEN,
        horizon=BASELINE_HORIZON
    ),

    "LSTM": SimpleLSTMForecaster(
        input_dim=BASELINE_INPUT_DIM,
        hidden_dim=LSTM_HIDDEN_DIM,
        num_layers=LSTM_NUM_LAYERS,
        horizon=BASELINE_HORIZON
    )
}

# 5. Validate forward passes
baseline_results = []

for name, model in baseline_models.items():

    model.eval()

    with torch.no_grad():

        prediction, z, z_next = model(
            baseline_input,
            return_latents=True
        )

        direct_prediction = model(baseline_input)

    expected_shape = (
        BASELINE_BATCH,
        BASELINE_INPUT_DIM * BASELINE_HORIZON
    )

    assert prediction.shape == expected_shape

    torch.testing.assert_close(
        prediction,
        direct_prediction
    )

    assert torch.isfinite(prediction).all()
    assert torch.isfinite(z).all()
    assert torch.isfinite(z_next).all()

    torch.testing.assert_close(z, z_next)

    if name == "LSTM":

        assert z.shape == (
            BASELINE_BATCH,
            LSTM_HIDDEN_DIM
        )

        assert model.lstm.num_layers == 2
        assert model.lstm.hidden_size == 96

    else:

        assert z.shape == expected_shape

        assert len(model.linears) == BASELINE_INPUT_DIM

        # Author's DLinear repeats the same value
        # for every forecasting step of each channel.
        pred_time_major = prediction.reshape(
            BASELINE_BATCH,
            BASELINE_HORIZON,
            BASELINE_INPUT_DIM
        )

        torch.testing.assert_close(
            pred_time_major,
            pred_time_major[:, :1, :].expand_as(
                pred_time_major
            )
        )

    parameters = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    baseline_results.append({
        "Model": name,
        "Input": tuple(baseline_input.shape),
        "Prediction": tuple(prediction.shape),
        "Latent": tuple(z.shape),
        "Parameters": parameters,
        "Status": "PASSED"
    })


# 6. Validate parameter counts
baseline_counts = {
    row["Model"]: row["Parameters"]
    for row in baseline_results
}

assert baseline_counts["DLinear"] == (
    BASELINE_INPUT_DIM * (BASELINE_PATCH_LEN + 1)
)

assert baseline_counts["LSTM"] == 117928


# 7. Final summary
print("=== BASELINE MODEL VALIDATION ===")

print("Input shape:", tuple(baseline_input.shape))
print("Patch length:", BASELINE_PATCH_LEN)
print("Horizon:", BASELINE_HORIZON)
print("Channels:", BASELINE_INPUT_DIM)

print("\n=== BASELINE MODELS ===")
display(pd.DataFrame(baseline_results))

print("\n=== MODEL DETAILS ===")
print("LSTM hidden dimension:", LSTM_HIDDEN_DIM)
print("LSTM layers:", LSTM_NUM_LAYERS)
print("DLinear channels:", len(
    baseline_models["DLinear"].linears
))

print("\nAll baseline forward passes validated!")
print("Cell 6 completed successfully!")


=== BASELINE MODEL VALIDATION ===
Input shape: (4, 120, 5)
Patch length: 120
Horizon: 8
Channels: 5

=== BASELINE MODELS ===


,Model,Input,Prediction,Latent,Parameters,Status
0,DLinear,"(4, 120, 5)","(4, 40)","(4, 40)",605,PASSED
1,LSTM,"(4, 120, 5)","(4, 40)","(4, 96)",117928,PASSED



=== MODEL DETAILS ===
LSTM hidden dimension: 96
LSTM layers: 2
DLinear channels: 5

All baseline forward passes validated!
Cell 6 completed successfully!


UNIFIED ABLATION MODEL FACTORY

In [7]:
# 1. Unified Model Factory
def create_ablation_model(
    model_name: str,
    input_dim: int,
    patch_len: int,
    horizon: int
):
    """
    Create a model according to the author's
    Ablation Study configuration.

    Returns:
        model: PyTorch model
        koop_attr: Koopman attribute or None
        lyap_weight: Lyapunov regularization weight
    """

    if model_name not in MODEL_VARIANTS:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    if min(input_dim, patch_len, horizon) <= 0:
        raise ValueError(
            "input_dim, patch_len and horizon must be positive."
        )

    koop_attr = None
    lyap_weight = 0.0

    # 1. DLinear
    if model_name == "DLinear":

        model = DLinearForecaster(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon
        )

    # 2. LSTM
    elif model_name == "LSTM":

        model = SimpleLSTMForecaster(
            input_dim=input_dim,
            hidden_dim=LSTM_HIDDEN_DIM,
            num_layers=LSTM_NUM_LAYERS,
            horizon=horizon
        )

    # 3. PatchTST Baseline
    elif model_name == "PatchTST":

        model = PatchTSTBaseline(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    # 4. PatchTST + Unconstrained Linear Latent
    elif model_name == "PatchTST+LinearLatent":

        model = PatchTST_UnconstrainedLatent(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    # 5. Koopman without Lyapunov regularization
    elif model_name == "KoopPatchTST_NoLyap":

        model = Koopformer_PatchTST(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

        koop_attr = "koop"
        lyap_weight = 0.0

    # 6. Full DeepKoopFormer
    elif model_name == "KoopPatchTST_Full":

        model = Koopformer_PatchTST(
            input_dim=input_dim,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

        koop_attr = "koop"
        lyap_weight = LYAP_WEIGHT

    return model, koop_attr, lyap_weight


# 2. Test configurations for all four datasets
factory_test_configs = {
    "Electricity": (70, 2),
    "Financial": (80, 4),
    "CMIP6 Wind": (120, 8),
    "CMIP6 Pressure": (130, 16)
}

# 3. Validate all six models on every dataset
factory_results = []

for dataset_name, (p, h) in factory_test_configs.items():

    bundle = ABLATION_DATASETS[dataset_name]
    F = bundle["input_dim"]

    dummy_input = torch.randn(2, p, F)

    for model_name in MODEL_VARIANTS:

        model, koop_attr, lyap_weight = (
            create_ablation_model(
                model_name=model_name,
                input_dim=F,
                patch_len=p,
                horizon=h
            )
        )

        model.eval()

        with torch.no_grad():

            pred, z, z_next = model(
                dummy_input,
                return_latents=True
            )

            direct_pred = model(dummy_input)

        # Forecast validation
        expected_shape = (2, F * h)

        assert pred.shape == expected_shape

        torch.testing.assert_close(
            pred,
            direct_pred,
            rtol=1e-5,
            atol=1e-5
        )

        assert torch.isfinite(pred).all()
        assert torch.isfinite(z).all()
        assert torch.isfinite(z_next).all()

        # Operator validation
        if model_name in [
            "KoopPatchTST_NoLyap",
            "KoopPatchTST_Full"
        ]:
            assert isinstance(
                model.koop,
                StrictStableKoopmanOperator
            )

            assert koop_attr == "koop"

        else:
            assert koop_attr is None

        # Lyapunov configuration
        expected_weight = (
            LYAP_WEIGHT
            if model_name == "KoopPatchTST_Full"
            else 0.0
        )

        assert lyap_weight == expected_weight

        # Parameter count
        n_params = sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )

        factory_results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "Input": tuple(dummy_input.shape),
            "Prediction": tuple(pred.shape),
            "Parameters": n_params,
            "LyapWeight": lyap_weight,
            "Status": "PASSED"
        })

        del model


# 4. Build validation table
factory_df = pd.DataFrame(factory_results)

assert len(factory_df) == 4 * 6
assert (factory_df["Status"] == "PASSED").all()

assert (
    factory_df.groupby("Dataset")["Model"].nunique()
    == 6
).all()

# 5. Verify that the two Koopman variants
#    have equal parameter counts for every dataset
for dataset_name in factory_test_configs:

    subset = factory_df[
        factory_df["Dataset"] == dataset_name
    ]

    no_lyap_params = subset.loc[
        subset["Model"] == "KoopPatchTST_NoLyap",
        "Parameters"
    ].iloc[0]

    full_params = subset.loc[
        subset["Model"] == "KoopPatchTST_Full",
        "Parameters"
    ].iloc[0]

    assert no_lyap_params == full_params


# 6. Final summary
print("=== UNIFIED MODEL FACTORY VALIDATION ===")

display(factory_df)

print("\n=== VALIDATION SUMMARY ===")
print("Datasets tested:", len(factory_test_configs))
print("Models per dataset:", len(MODEL_VARIANTS))
print("Total forward-pass tests:", len(factory_df))

print("\n=== LYAPUNOV WEIGHTS ===")

for name in MODEL_VARIANTS:

    _, _, weight = create_ablation_model(
        model_name=name,
        input_dim=5,
        patch_len=120,
        horizon=8
    )

    print(f"{name:<28} lambda = {weight}")

print("\nAll 6 models initialized successfully.")
print("All 24 model-dataset tests passed.")
print("Cell 7 completed successfully!")


=== UNIFIED MODEL FACTORY VALIDATION ===


,Dataset,Model,Input,Prediction,Parameters,LyapWeight,Status
0,Electricity,DLinear,"(2, 70, 6)","(2, 12)",426,0.0,PASSED
1,Electricity,LSTM,"(2, 70, 6)","(2, 12)",115596,0.0,PASSED
2,Electricity,PatchTST,"(2, 70, 6)","(2, 12)",210444,0.0,PASSED
3,Electricity,PatchTST+LinearLatent,"(2, 70, 6)","(2, 12)",219660,0.0,PASSED
4,Electricity,KoopPatchTST_NoLyap,"(2, 70, 6)","(2, 12)",228972,0.0,PASSED
5,Electricity,KoopPatchTST_Full,"(2, 70, 6)","(2, 12)",228972,0.1,PASSED
6,Financial,DLinear,"(2, 80, 2)","(2, 8)",162,0.0,PASSED
7,Financial,LSTM,"(2, 80, 2)","(2, 8)",113672,0.0,PASSED
8,Financial,PatchTST,"(2, 80, 2)","(2, 8)",185096,0.0,PASSED
9,Financial,PatchTST+LinearLatent,"(2, 80, 2)","(2, 8)",194312,0.0,PASSED



=== VALIDATION SUMMARY ===
Datasets tested: 4
Models per dataset: 6
Total forward-pass tests: 24

=== LYAPUNOV WEIGHTS ===
DLinear                      lambda = 0.0
LSTM                         lambda = 0.0
PatchTST                     lambda = 0.0
PatchTST+LinearLatent        lambda = 0.0
KoopPatchTST_NoLyap          lambda = 0.0
KoopPatchTST_Full            lambda = 0.1

All 6 models initialized successfully.
All 24 model-dataset tests passed.
Cell 7 completed successfully!


GRADIENT & TRAINING READINESS VALIDATION

In [8]:
import torch.nn.functional as F

# 1. Use real training data from CMIP6 Wind
TEST_DATASET = "CMIP6 Wind"
TEST_PATCH_LEN = 70
TEST_HORIZON = 8
TEST_BATCH_SIZE = 4

train_series = ABLATION_DATASETS[TEST_DATASET]["train"]

input_dim = train_series.shape[1]

# Build a small batch using the author's window convention
x_batch = torch.stack([
    train_series[i:i + TEST_PATCH_LEN]
    for i in range(TEST_BATCH_SIZE)
]).to(DEVICE)

y_batch = torch.stack([
    train_series[
        i + TEST_PATCH_LEN:
        i + TEST_PATCH_LEN + TEST_HORIZON
    ].T.reshape(-1)
    for i in range(TEST_BATCH_SIZE)
]).to(DEVICE)

assert x_batch.shape == (
    TEST_BATCH_SIZE, TEST_PATCH_LEN, input_dim
)

assert y_batch.shape == (
    TEST_BATCH_SIZE, input_dim * TEST_HORIZON
)

# 2. Validate one optimizer step for each model
training_results = []

for model_name in MODEL_VARIANTS:

    # Initialize a fresh model
    model, koop_attr, lyap_weight = create_ablation_model(
        model_name=model_name,
        input_dim=input_dim,
        patch_len=TEST_PATCH_LEN,
        horizon=TEST_HORIZON
    )

    model = model.to(DEVICE)
    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # Snapshot before training
    initial_params = {
        name: param.detach().clone()
        for name, param in model.named_parameters()
    }

    # 3. Forward pass
    optimizer.zero_grad()

    pred, z, z_next = model(
        x_batch,
        return_latents=True
    )

    pred_loss = F.mse_loss(pred, y_batch)

    # 4. Lyapunov regularization
    if lyap_weight > 0:

        P = torch.eye(
            z.shape[1],
            device=DEVICE
        )

        zp = torch.einsum(
            "bi,ij->bj", z, P
        )

        znp = torch.einsum(
            "bi,ij->bj", z_next, P
        )

        lyap_loss = torch.relu(
            (znp * z_next).sum(dim=1)
            - (zp * z).sum(dim=1)
        ).mean()

    else:
        lyap_loss = torch.zeros(
            (), device=DEVICE
        )

    total_loss = (
        pred_loss
        + lyap_weight * lyap_loss
    )

    assert torch.isfinite(total_loss).item()

    # 5. Backpropagation
    total_loss.backward()

    gradient_norm_sq = 0.0
    params_with_gradient = 0

    for param in model.parameters():

        if param.grad is not None:

            assert torch.isfinite(
                param.grad
            ).all().item()

            gradient_norm_sq += (
                param.grad.detach().norm().item() ** 2
            )

            params_with_gradient += 1

    gradient_norm = gradient_norm_sq ** 0.5

    assert gradient_norm > 0
    assert params_with_gradient > 0

    # 6. Update parameters
    optimizer.step()

    updated_params = 0

    for name, param in model.named_parameters():

        if not torch.equal(
            initial_params[name],
            param.detach()
        ):
            updated_params += 1

        assert torch.isfinite(param).all().item()

    assert updated_params > 0

    # 7. Koopman stability after optimizer step
    max_singular_value = None

    if isinstance(
        getattr(model, "koop", None),
        StrictStableKoopmanOperator
    ):

        model.eval()

        with torch.no_grad():
            _, K, sigma = model.koop(z.detach())

            max_singular_value = (
                torch.linalg.svdvals(K)
                .max()
                .item()
            )

        assert max_singular_value < RHO_MAX + 1e-5

    # 8. Record results
    training_results.append({
        "Model": model_name,
        "PredictionLoss": pred_loss.item(),
        "LyapunovLoss": lyap_loss.item(),
        "TotalLoss": total_loss.item(),
        "GradientNorm": gradient_norm,
        "UpdatedParams": updated_params,
        "MaxSingularValue": max_singular_value,
        "Status": "PASSED"
    })

    del model, optimizer

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# 9. Final validation
training_df = pd.DataFrame(training_results)

assert len(training_df) == 6
assert (training_df["Status"] == "PASSED").all()
assert (training_df["GradientNorm"] > 0).all()
assert (training_df["UpdatedParams"] > 0).all()

# 10. Print summary
print("=== TRAINING READINESS VALIDATION ===")

print("Dataset:", TEST_DATASET)
print("Device:", DEVICE)
print("Input:", tuple(x_batch.shape))
print("Target:", tuple(y_batch.shape))
print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)

print("\n=== GRADIENT & OPTIMIZER TEST ===")
display(training_df)

print("\n=== VALIDATION SUMMARY ===")
print("Models tested:", len(training_df))
print("Models with valid gradients:",
      int((training_df["GradientNorm"] > 0).sum()))
print("Models with updated parameters:",
      int((training_df["UpdatedParams"] > 0).sum()))

print("\nAll gradients are finite.")
print("All six models updated their parameters.")
print("Koopman stability preserved after optimizer step.")
print("All models are ready for training!")
print("Cell 8 completed successfully!")


=== TRAINING READINESS VALIDATION ===
Dataset: CMIP6 Wind
Device: cuda
Input: (4, 70, 5)
Target: (4, 40)
Optimizer: Adam
Learning rate: 0.0003

=== GRADIENT & OPTIMIZER TEST ===


,Model,PredictionLoss,LyapunovLoss,TotalLoss,GradientNorm,UpdatedParams,MaxSingularValue,Status
0,DLinear,0.079662,0.0,0.079662,0.692481,10,NaN,PASSED
1,LSTM,0.052274,0.0,0.052274,0.084967,10,NaN,PASSED
2,PatchTST,0.368291,0.0,0.368291,2.356118,41,NaN,PASSED
3,PatchTST+LinearLatent,0.113292,0.0,0.113292,0.974303,42,NaN,PASSED
4,KoopPatchTST_NoLyap,0.171256,0.0,0.171256,0.906705,44,0.891027,PASSED
5,KoopPatchTST_Full,0.130916,0.0,0.130916,0.787802,44,0.915752,PASSED



=== VALIDATION SUMMARY ===
Models tested: 6
Models with valid gradients: 6
Models with updated parameters: 6

All gradients are finite.
All six models updated their parameters.
Koopman stability preserved after optimizer step.
All models are ready for training!
Cell 8 completed successfully!
